<a href="https://colab.research.google.com/github/LionPaul/retro-game-miner/blob/main/snes/notebooks/01_bronze_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
from io import StringIO
import os

# ==============================================================================
# PIPELINE CONFIGURATION (Easily adaptable for other consoles)
# ==============================================================================
CONSOLE_NAME = "snes"
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_Super_Nintendo_Entertainment_System_games"

# Padronização numérica sequencial para os datasets (Sua nova convenção)
DATASETS_DIR = "datasets"
OUTPUT_FILENAME = f"1_{CONSOLE_NAME}_games_raw.xlsx" # Alterado para manter o padrão '1_console_...'
OUTPUT_PATH = os.path.join(DATASETS_DIR, OUTPUT_FILENAME)
# ==============================================================================

def pipeline_extract_bronze(url, output_path):
    """
    Extracts the complete and raw list of games from Wikipedia.
    Acts as the Bronze Layer of the Medallion Architecture.
    """
    print(f"🎬 Starting RAW data extraction for console: {CONSOLE_NAME.upper()}")

    # Cria o diretório de dados se não existir
    if not os.path.exists(DATASETS_DIR):
        os.makedirs(DATASETS_DIR)
        print(f"📁 Directory '{DATASETS_DIR}' created successfully.")

    # User-Agent atualizado para emular um navegador real e evitar bloqueios (HTTP 403)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    }

    try:
        # 1. Requisição HTTP
        print(f"🌐 Fetching HTML from Wikipedia: {url}")
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        print("✅ HTML download completed successfully.")

        # 2. Parsing das tabelas HTML
        html_data = StringIO(response.text)
        tables = pd.read_html(html_data)

        # 3. Identificação do catálogo principal (Algoritmo heurístico baseado em volume de linhas)
        print("📊 Scanning tables to identify the main game catalog...")
        df_games = max(tables, key=len)
        print(f"🎯 Target table found with {len(df_games)} raw records.")

        # 4. Tratamento de MultiIndex (Headers aninhados/duplos da Wikipedia)
        if isinstance(df_games.columns, pd.MultiIndex):
            print("🔗 Flattening MultiIndex table headers...")
            df_games.columns = [' '.join(col).strip() for col in df_games.columns.values]

        # 5. Persistência do dado bruto (Sem filtros destrutivos na Camada 1)
        print(f"💾 Saving raw data into the Bronze layer...")
        df_games.to_excel(output_path, index=False)
        print(f"🎉 SUCCESS! Dataset saved at: {output_path}")

        # Retorna o preview para controle interno no Colab
        return df_games.head(5)

    except Exception as e:
        print(f"❌ Critical error during extraction pipeline: {e}")
        return None

# Execução do processo
df_raw_preview = pipeline_extract_bronze(WIKI_URL, OUTPUT_PATH)

🎬 Iniciando a extração dos dados brutos para o console: SNES
📁 Pasta 'datasets' criada com sucesso.
🌐 Conectando à Wikipedia em: https://en.wikipedia.org/wiki/List_of_Super_Nintendo_Entertainment_System_games
✅ Download do HTML concluído com sucesso.
📊 Analisando tabelas da página para encontrar o catálogo de jogos...
🎯 Tabela principal identificada com 1749 registros brutos.
🔗 Combinando subcabeçalhos duplicados da tabela...
💾 Salvando os dados brutos na camada Bronze...
🎉 SUCESSO! Arquivo salvo em: datasets/snes_games_raw.xlsx
